# Kumo Engine task examples with TPC-H on Databricks

Explore binary classification, regression, and link prediction with Kumo Engine on TPC-H data. The notebook trains and scores the binary task by default; the other query blocks are commented alternatives.

This walkthrough defaults to a **classic Databricks GPU cluster with dedicated access mode**. The same notebook can be adapted for Serverless GPU by switching the scratch path and enabling the commented training workaround below. It reads the built-in `samples.tpch` dataset, materializes a local graph snapshot, and retains model bundles and diagnostics in a Unity Catalog Volume. No external credentials are required.

Each user is responsible for checking the content of datasets and the applicable licenses and determining if suitable for the intended use


## Prepare your environment

**Runtime choices:** the default path is a classic all-purpose GPU cluster with dedicated access mode on Databricks Runtime 17.3 LTS ML (Python 3.12.3). For Serverless GPU, select a Python 3.12 AI Runtime GPU environment, use the commented `/tmp/kumoai-local` scratch setting, and enable the commented `spawn` workaround only if Optuna child processes encounter IPC authentication errors. Serverless package/runtime compatibility has not been verified end-to-end here. In both cases, GPU training runs on notebook compute; a separate SQL warehouse identified by `SQL_WAREHOUSE_ID` serves the connector's SQL work.

**Compute and storage:** choose a CUDA-capable GPU and sufficient CPU RAM/GPU memory for the graph and model. There is no verified universal GPU size or exact disk requirement for this workload. Reserve at least four times the materialized graph-index size in free local scratch, plus room for retained runs and ZIP creation. Use `/local_disk0/tmp/kumoai-local` for ephemeral scratch on classic dedicated compute, or the commented `/tmp/kumoai-local` path on Serverless GPU; `/local_disk0` is not writable on serverless or standard access mode. `/tmp` is temporary and should not hold durable or sensitive exports. Use a pre-created Unity Catalog Volume under `/Volumes/<catalog>/<schema>/<volume>` for durable model bundles and diagnostics.

**Permissions and access:** attach/use the chosen GPU compute; install `kumo-engine[databricks]` from PyPI; use the SQL warehouse (`CAN USE`); read `samples.tpch` (`USE CATALOG`, `USE SCHEMA`, `SELECT`); create/replace views in the destination catalog/schema (`USE CATALOG`, `USE SCHEMA`, `CREATE TABLE`, plus `CREATE SCHEMA` on the catalog if this notebook creates that schema); and read/write the configured Volume (`READ VOLUME`, `WRITE VOLUME`). Enabling prediction output also requires permission to create or overwrite its dedicated table. Existing views must be owned or manageable by the running identity.

See the [Databricks Runtime 17.3 LTS ML release notes](https://docs.databricks.com/aws/en/release-notes/runtime/17.3lts-ml) and [Databricks local-file guidance](https://docs.databricks.com/aws/en/files/write-data) for current platform details.

The TPC-H dataset contains `customer`, `orders`, `lineitem`, and `part` with the columns used below. The notebook materializes local graph data and adds a synthetic primary key for `lineitem`.

## 1. Enter your configuration

Set the destination Unity Catalog catalog and schema for the views and output. The sample source remains `samples.tpch`. Kumo uses the configured local scratch directory to materialize graph and training data. A Unity Catalog Volume retains model bundles and optional diagnostic archives. Set the prediction date within the data's date range (1992-01-01 to 1998-08-02).


In [ ]:
from datetime import datetime
from pathlib import Path

# ── TPC-H source (Databricks sample data) ──
TPCH_CATALOG = "samples"
TPCH_SCHEMA = "tpch"

# ── Databricks connector ──
CONNECTOR_CATALOG = "<catalog>"                      # Your Unity Catalog catalog
CONNECTOR_SCHEMA = "<schema>"                        # Schema for TPC-H views and output
SQL_WAREHOUSE_ID = "<warehouse_id>"                  # SQL warehouse for the DatabricksConnector

# ── Local storage and model settings ──
# Classic GPU with dedicated access mode (default):
SCRATCH_DIR = "/local_disk0/tmp/kumoai-local"
# Serverless GPU: comment out the classic line above and uncomment this local temporary path.
# SCRATCH_DIR = "/tmp/kumoai-local"
ARTIFACT_ROOT = Path("/Volumes/<catalog>/<schema>/<volume>/tpch-runs")
MODEL_NAME = "TPCH_CUSTOMER_ORDER_MODEL"
PREDICTION_TIME = datetime(1998, 1, 1)              # Orders range 1992-01-01 to 1998-08-02.
PREDICTION_OUTPUT_TABLE = "TPCH_ORDER_PREDICTIONS"
SAVED_ARCHIVE = None                                # Set a prior ZIP path under ARTIFACT_ROOT to inspect it.

# Optional actions. Prediction output is overwritten when enabled.
INSPECT_DIAGNOSTICS = True
PLOT_LEARNING_CURVES = True
WRITE_PREDICTIONS = True
UPLOAD_DIAGNOSTICS = True
DOWNLOAD_DIAGNOSTICS = True
RUN_INDEX = None
PLAN_INDEX = 0


## 2. Install Kumo Engine from PyPI

Run the installation cell once, then restart Python. After the restart, rerun configuration and continue at imports.


In [ ]:
%pip install "kumo-engine[databricks]"

dbutils.library.restartPython()


<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and these cells. They do not depend on variables created during installation. Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
import json
from importlib.metadata import version
import shutil
import tempfile
from uuid import uuid4

import pandas as pd
from IPython.display import display

import kumo.engine as kumo
from kumoapi.typing import Dtype, Stype
from kumoapi.graph import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode


In [ ]:
# Create local scratch and retained-artifact directories, test write access, configure Kumo, and verify GPU access.
import sys
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Use Python 3.12; found {sys.version.split()[0]}.")
scratch_root = Path(SCRATCH_DIR).expanduser().resolve()
if not (
    scratch_root.is_relative_to(Path("/local_disk0"))
    or scratch_root.is_relative_to(Path("/tmp"))
):
    raise ValueError("Use /local_disk0 for classic dedicated compute or /tmp for Serverless GPU.")
if not str(ARTIFACT_ROOT).startswith("/Volumes/") or "<" in str(ARTIFACT_ROOT):
    raise ValueError("Set ARTIFACT_ROOT to an existing writable Unity Catalog Volume path.")
if not ARTIFACT_ROOT.parent.is_dir():
    raise FileNotFoundError("Create the Unity Catalog Volume before running this notebook.")
scratch_root.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / "staging").mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch_root) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
with tempfile.TemporaryFile(dir=ARTIFACT_ROOT) as probe:
    probe.write(b"artifact volume check")
    probe.flush()

kumo.configure(scratch_dir=str(scratch_root))
print("Python:", sys.version.split()[0])
print("Kumo version:", kumo.__version__)
print("GA package:", version("kumo-engine"))
print("Scratch:", scratch_root)
print("Retained artifacts:", ARTIFACT_ROOT)
print("Available scratch GiB:", round(shutil.disk_usage(scratch_root).free / 2**30, 1))

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Use a Databricks GPU cluster with a compatible runtime.")
for index in range(torch.cuda.device_count()):
    device = torch.cuda.get_device_properties(index)
    print("GPU:", device.name, "VRAM GiB:", round(device.total_memory / 2**30, 1))


The printed version identifies the installed build. The probe checks write access and free space at this path; it does not establish that the path is on supported local storage. Confirm the backing disk and capacity before materializing data.

## Connect to TPC-H data

The notebook creates Unity Catalog views of the four TPC-H sample tables in the configured schema. The `lineitem` view adds a synthetic single-column primary key (`lineitem_id`) since TPC-H uses a composite key. A `DatabricksConnector` uses the notebook's default auth chain — no separate credentials are needed.


In [ ]:
# ── Create TPC-H views in the connector schema ──
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}")

for table in ["customer", "orders", "part"]:
    spark.sql(f"""
        CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{table} AS
        SELECT * FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.{table}
    """)
    print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{table}")

# Lineitem: add synthetic single-column primary key (composite key not supported)
spark.sql(f"""
    CREATE OR REPLACE VIEW {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.lineitem AS
    SELECT *, CAST(l_orderkey * 8 + l_linenumber AS BIGINT) AS lineitem_id
    FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.lineitem
""")
print(f"View: {CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.lineitem (with lineitem_id)")

# ── Create Kumo Databricks connector (uses notebook's default auth) ──
connector = kumo.DatabricksConnector.from_notebook(
    catalog=CONNECTOR_CATALOG,
    schema_name=CONNECTOR_SCHEMA,
    warehouse_id=SQL_WAREHOUSE_ID,
)
print("Tables:", connector.table_names())


## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value. `with_column()` returns an updated definition.

The customer and part definitions select the columns used in this example. TPC-H column names use the standard `c_`, `o_`, `l_`, `p_` prefixes.

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["customer"],
        ["c_custkey", "c_nationkey", "c_mktsegment", "c_acctbal"],
        primary_key="c_custkey",
    )
    .with_column("c_nationkey", stype=Stype.categorical)
    .with_column("c_mktsegment", stype=Stype.categorical)
    .with_column("c_acctbal", stype=Stype.numerical)
)

part = (
    kumo.Table.from_source_table(
        connector["part"],
        ["p_partkey", "p_name", "p_brand", "p_type", "p_size",
         "p_container", "p_retailprice"],
        primary_key="p_partkey",
    )
    .with_column("p_name", stype=Stype.text)
    .with_column("p_brand", stype=Stype.categorical)
    .with_column("p_type", stype=Stype.categorical)
    .with_column("p_size", stype=Stype.numerical)
    .with_column("p_container", stype=Stype.categorical)
    .with_column("p_retailprice", stype=Stype.numerical)
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

This sample uses `o_orderdate` for orders and `l_shipdate` for line items. The ship date represents when the item became observable in the fulfillment process and is a reasonable availability timestamp for this dataset.

In [ ]:
orders = (
    kumo.Table.from_source_table(
        connector["orders"],
        ["o_orderkey", "o_custkey", "o_orderdate", "o_orderstatus",
         "o_totalprice", "o_orderpriority", "o_shippriority"],
        primary_key="o_orderkey",
        time_column="o_orderdate",
    )
    .with_column("o_custkey", stype=Stype.ID)
    .with_column("o_orderdate", dtype=Dtype.date)
    .with_column("o_orderstatus", stype=Stype.categorical)
    .with_column("o_totalprice", stype=Stype.numerical)
    .with_column("o_orderpriority", stype=Stype.categorical)
    .with_column("o_shippriority", stype=Stype.numerical)
)

In [ ]:
lineitem = (
    kumo.Table.from_source_table(
        connector["lineitem"],
        ["lineitem_id", "l_orderkey", "l_partkey", "l_shipdate",
         "l_quantity", "l_extendedprice", "l_discount", "l_tax",
         "l_returnflag", "l_linestatus", "l_shipmode"],
        primary_key="lineitem_id",
        time_column="l_shipdate",
    )
    .with_column("l_orderkey", stype=Stype.ID)
    .with_column("l_partkey", stype=Stype.ID)
    .with_column("l_shipdate", dtype=Dtype.date)
    .with_column("l_quantity", stype=Stype.numerical)
    .with_column("l_extendedprice", stype=Stype.numerical)
    .with_column("l_discount", stype=Stype.numerical)
    .with_column("l_tax", stype=Stype.numerical)
    .with_column("l_returnflag", stype=Stype.categorical)
    .with_column("l_linestatus", stype=Stype.categorical)
    .with_column("l_shipmode", stype=Stype.categorical)
)

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
lineitem.l_partkey   -> part.p_partkey
lineitem.l_orderkey  -> orders.o_orderkey
orders.o_custkey     -> customer.c_custkey
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:
graph = kumo.Graph(
    tables=[customer, part, orders, lineitem],
    edges=[
        Edge(src_table="orders", fkey="o_custkey", dst_table="customer"),
        Edge(src_table="lineitem", fkey="l_orderkey", dst_table="orders"),
        Edge(src_table="lineitem", fkey="l_partkey", dst_table="part"),
    ],
)

graph.validate()
print("graph id:", graph.id)

In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)
print("Local index:", snapshot.index_path)

## Define the prediction question

Predictive Query Language (PQL) is Kumo Engine's declarative syntax for defining a predictive modeling task: it specifies the target, the entity to score, and the data context. This default query predicts whether a customer who placed an order in the previous year will place another order in the next 90 days. The `WHERE` clause selects that active-customer population, and `> 0` makes the target binary.

In [ ]:
# Binary classification: predict whether a customer active in the past year will order again in the next 90 days.
pquery = kumo.PredictiveQuery(
    graph=graph,
    query="""
        PREDICT COUNT(orders.*, 0, 90, days) > 0
        FOR EACH customer.c_custkey
        WHERE COUNT(orders.*, -365, 0, days) > 0
    """,
)

pquery.validate()
print("Task type:", pquery.get_task_type())

### Optional task alternatives

The next two code cells are commented out. To try one, skip or comment out the binary query cell above, uncomment exactly one alternative, and run it before training. Both define `pquery` and use the existing `graph`, `snapshot`, and downstream variables. Train a new model for the selected task. The link example ranks parts associated with orders, not customer-level recommendations.

In [ ]:
# Regression: predict how many orders a customer active in the past year will place in the next 90 days.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT COUNT(orders.*, 0, 90, days) "
#         "FOR EACH customer.c_custkey "
#         "WHERE COUNT(orders.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

In [ ]:
# Link prediction: rank up to five parts that may ship in the next 90 days for an order with recent line items.
# pquery = kumo.PredictiveQuery(
#     graph=graph,
#     query=(
#         "PREDICT LIST_DISTINCT(lineitem.l_partkey, 0, 90, days) "
#         "RANK TOP 5 FOR EACH orders.o_orderkey "
#         "WHERE COUNT(lineitem.*, -365, 0, days) > 0"
#     ),
# )
# pquery.validate()
# print("Task type:", pquery.get_task_type())

### Generate a Training Table

After your query is validated, generate a training table to use for model fitting.

In [ ]:
train_table = pquery.generate_training_table(snapshot)
print("Training table:", train_table.train_table_path)


## Create a Trainer

The suggested model plan controls architecture, sampling, and optimization. The model plan is fully customizable: inspect and adjust its settings for your task, then create `trainer = kumo.Trainer(model_plan)` before fitting.

In [ ]:
model_plan = pquery.suggest_model_plan(snapshot)
model_plan.optimization.max_epochs = 2
# Create a Trainer with your model plan.
trainer = kumo.Trainer(model_plan)
print(model_plan)
print("Validation selection metric:", getattr(
    model_plan.training_job, "tune_metric", "not exposed by this build"))


### Train a Model

After creating a Trainer from the model plan, call `fit()` with the training table and compatible graph snapshot. This cell then copies the complete model bundle to the configured Unity Catalog Volume. Each run receives a unique retained path. Running the cell again starts another fit.


In [ ]:
runs_before_fit = {p.parent for p in scratch_root.rglob("auto_train_progress.json")}

# Serverless GPU only: if Optuna child processes hit IPC authentication errors,
# uncomment this block before fit(). Keep it commented on classic GPU compute.
# import multiprocessing
# try:
#     multiprocessing.set_start_method("spawn", force=True)
# except RuntimeError:
#     pass

import warnings
from optuna.exceptions import ExperimentalWarning

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"`warn_independent_sampling` has been deprecated.*",
        category=FutureWarning,
    )
    warnings.filterwarnings(
        "ignore",
        message=r"Argument ``multivariate`` is an experimental feature.*",
        category=ExperimentalWarning,
    )
    warnings.filterwarnings(
        "ignore",
        message=r"Setting an item of incompatible dtype is deprecated.*",
        category=FutureWarning,
    )
    training_result = trainer.fit(train_table, snapshot=snapshot)

run_root = ARTIFACT_ROOT / MODEL_NAME / uuid4().hex
run_root.mkdir(parents=True, exist_ok=False)
shutil.copytree(Path(training_result.bundle_dir), run_root / "model")
print("Retained model bundle:", run_root / "model")


### Inspect the selected result

Review the reported metrics and their stage names. Available metrics vary by task and build. The retained model bundle preserves model state; it does not archive the complete experiment history or explanations.


In [ ]:
print("Available metrics:", training_result.metrics)
print("Reusable local bundle:", training_result.bundle_dir)
for name, path in training_result.artifact_paths.items():
    print(name, path)


<a id="prediction"></a>
## Generate Batch Predictions

After training a model, generate batch predictions and write them to a Unity Catalog table. `WRITE_PREDICTIONS = True` enables that write by default; set it to `False` for local-only results.

Check the TPC-H order-date range against `PREDICTION_TIME` before scoring. A chosen date alone does not guarantee the following 90 days of observed outcomes for historical evaluation.

In [ ]:
date_range = spark.sql(f"""
    SELECT MIN(o_orderdate) AS first_order, MAX(o_orderdate) AS last_order
    FROM {TPCH_CATALOG}.{TPCH_SCHEMA}.orders
""")
display(date_range.toPandas())
print("Configured prediction boundary:", PREDICTION_TIME)


### Generate the scoring population

For these temporal queries, use a timezone-naive timestamp interpreted as UTC and reuse the fitted trainer's saved training-table generation plan. Binary classification and regression score customers with an order in the past year; the link example scores orders with recent line items. This timestamp example does not apply to static queries.

In [ ]:
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)

### Score locally, with an optional Unity Catalog output

**`WRITE_PREDICTIONS = True` authorizes `OVERWRITE`, which replaces the configured destination table's contents.** Choose a dedicated Unity Catalog table you own, never a source table. Set `WRITE_PREDICTIONS = False` to keep results only in local scratch.


In [ ]:
prediction_output = None
if WRITE_PREDICTIONS:
    import re
    from kumo_databricks.output import SEPARATOR as _SEP
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREDICTION_OUTPUT_TABLE):
        raise ValueError("Choose a simple dedicated output table name in configuration.")
    source_names = {"CUSTOMER", "PART", "ORDERS", "LINEITEM"}
    if PREDICTION_OUTPUT_TABLE.upper() in source_names:
        raise ValueError("Prediction output must not replace a source table.")
    prediction_output = kumo.PredictionOutputConfig(
        connector=connector, artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=f"{CONNECTOR_SCHEMA}{_SEP}{PREDICTION_OUTPUT_TABLE}",
        write_mode=WriteMode.OVERWRITE,
    )
from kumoapi.task import TaskType

prediction_options = {}
if pquery.get_task_type() == TaskType.BINARY_CLASSIFICATION:
    prediction_options["binary_classification_threshold"] = 0.5
elif pquery.get_task_type() == TaskType.TEMPORAL_LINK_PREDICTION:
    prediction_options["num_classes_to_return"] = 5
prediction_result = trainer.predict(
    prediction_table, snapshot=snapshot,
    output_config=prediction_output, **prediction_options,
)
print("Rows:", prediction_result.num_rows)
print("Local Parquet files:", prediction_result.files)
if WRITE_PREDICTIONS:
    print("Output:",
          f"{CONNECTOR_CATALOG}.{CONNECTOR_SCHEMA}.{PREDICTION_OUTPUT_TABLE}")


### Read a bounded preview

Read at most ten rows across the local Parquet files in batches. This avoids collecting the entire result. Parquet readers still need file metadata and decoding buffers. `prediction_result.frame` instead loads all rows into memory, even if followed by `.head()`.

In [ ]:
import pyarrow.parquet as pq

def preview_parquet(files, limit=10):
    frames = []
    remaining = limit
    for path in files:
        if remaining <= 0:
            break
        with pq.ParquetFile(path) as parquet:
            for batch in parquet.iter_batches(batch_size=remaining):
                frames.append(batch.slice(0, remaining).to_pandas())
                remaining -= len(frames[-1])
                if remaining <= 0:
                    break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

display(preview_parquet(prediction_result.files))

## Optional diagnostics

Inspection is enabled by default and is not required for prediction. Discovery lists local AutoML directories. The notebook automatically selects a run only when exactly one new directory appeared during this fit. If selection is ambiguous, set `RUN_INDEX` from the displayed list. Set `INSPECT_DIAGNOSTICS = False` to skip inspection.


In [ ]:
automl_dirs = sorted({
    path.parent for path in scratch_root.rglob("auto_train_progress.json")
    if path.is_file()
})
for index, path in enumerate(automl_dirs):
    print(index, path)
if not automl_dirs:
    print("No retained AutoML runs found under configured scratch.")

In [ ]:
automl_dir = None
if INSPECT_DIAGNOSTICS:
    if RUN_INDEX is not None:
        if not isinstance(RUN_INDEX, int) or not 0 <= RUN_INDEX < len(automl_dirs):
            raise ValueError("Choose one of the displayed run indices.")
        automl_dir = automl_dirs[RUN_INDEX]
    elif "runs_before_fit" in globals():
        new_runs = [path for path in automl_dirs if path not in runs_before_fit]
        if len(new_runs) == 1:
            automl_dir = new_runs[0]
    if automl_dir is None:
        print("No unambiguous run selected. Set RUN_INDEX from the displayed list.")
    else:
        print("Selected:", automl_dir)
else:
    print("Diagnostic inspection disabled.")


### Read progress and compare trials

These small readers report missing or unreadable files without blocking prediction. The progress summary shows completed/total trials and each trial's recorded epochs, elapsed seconds, and accelerator. Missing optional values mean they were not recorded.

In [ ]:
def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError) as error:
        print(f"Cannot read {path.name}: {error}")
        return {}

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (OSError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
        print(f"Cannot read {path.name}: {error}")
        return pd.DataFrame()

def history_frames(progress):
    summaries, curves = [], []
    for trial, details in (progress.get("trial_progress") or {}).items():
        details = details or {}
        summaries.append({"trial": str(trial),
                          "epochs_completed": details.get("num_epochs_completed"),
                          "elapsed_seconds": details.get("elapsed_time"),
                          "accelerator": details.get("accelerator")})
        for epoch, metrics in (details.get("metrics") or {}).items():
            try:
                epoch_number = int(epoch)
            except (TypeError, ValueError):
                print("Skipping unrecognized epoch:", epoch)
                continue
            metrics = metrics or {}
            curves.append({"trial": str(trial), "epoch": epoch_number,
                           **(metrics.get("train_metrics") or {}),
                           **(metrics.get("validation_metrics") or {})})
    summary = pd.DataFrame(summaries)
    history = pd.DataFrame(curves)
    if not history.empty:
        history = history.sort_values(["trial", "epoch"])
    return summary, history

In [ ]:
progress = read_json(automl_dir / "auto_train_progress.json") if automl_dir else {}
trial_summary, learning_curves = history_frames(progress)
print("Completed / total trials:", progress.get("completed_trials", "not recorded"),
      "/", progress.get("total_trials", "not recorded"))
if not trial_summary.empty:
    display(trial_summary)
else:
    print("No trial progress recorded or no run selected.")
if not learning_curves.empty:
    display(learning_curves)
print("Available epoch metrics:", [
    c for c in learning_curves.columns if c not in {"trial", "epoch"}
])

In [ ]:
trial_frames = []
for path in sorted(automl_dir.glob("version_*/val_result.csv")) if automl_dir else []:
    frame = read_csv(path)
    if not frame.empty:
        frame["trial_directory"] = path.parent.name
        trial_frames.append(frame)
if trial_frames:
    display(pd.concat(trial_frames, ignore_index=True))
else:
    print("No completed trial validation results found.")

plan_paths = sorted(automl_dir.glob("version_*/model_plan.json")) if automl_dir else []
for index, path in enumerate(plan_paths):
    print(index, path.relative_to(automl_dir))

The first available trial plan is displayed by default. Change `PLAN_INDEX` in configuration to inspect another. Compare trial validation results using the configured selection metric, not an assumed metric or direction.

In [ ]:
if plan_paths and PLAN_INDEX is not None:
    if not isinstance(PLAN_INDEX, int) or not 0 <= PLAN_INDEX < len(plan_paths):
        raise ValueError("Choose a displayed plan index.")
    display(read_json(plan_paths[PLAN_INDEX]))
else:
    print("No trial plan selected.")

### Plot available learning curves

Select exact metric names printed above. The default tries training and validation loss. Compare how validation changes while training improves. Divergence can suggest overfitting; nearly perfect results may warrant a leakage check. These are diagnostic signals, not proof of a cause. Plotting is optional and does not affect other inspection cells.

In [ ]:
METRIC_COLUMNS = ["train_loss", "val_loss"]
columns = [name for name in METRIC_COLUMNS if name in learning_curves.columns]
if not PLOT_LEARNING_CURVES:
    print("Learning-curve plotting disabled.")
elif learning_curves.empty or not columns:
    print("No matching completed epoch metrics. Choose from the available names.")
else:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Matplotlib is unavailable. Use the displayed epoch table instead.")
    else:
        figure, axis = plt.subplots(figsize=(9, 5))
        for trial, rows in learning_curves.groupby("trial", sort=True):
            for column in columns:
                values = pd.to_numeric(rows[column], errors="coerce")
                if values.notna().any():
                    axis.plot(rows["epoch"], values, marker="o", label=f"{trial}: {column}")
        axis.set(xlabel="Epoch", ylabel="Metric", title="Learning curves")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend()
        plt.show()

### Inspect gradient scores and column analysis

Post-fit gradient scoring and column analysis availability depends on the installed GA package, task, and model. Explanations can be disabled through the model plan's `training_job.disable_explain` setting.

Gradient scores describe model sensitivity to input columns, not causal effects. Column analysis groups feature values into cohorts and summarizes predictions and outcomes. Explanation errors may be logged while the model remains usable. Missing scores do not prove a column was excluded from training.

The following cells search beneath the selected run, display the actual artifact paths, and read what exists. They do not use a separate explanation API or require the plotting cell.

In [ ]:
score_paths = sorted(automl_dir.rglob("explain_test/scores.csv")) if automl_dir else []
if not score_paths:
    print("No gradient scores. Check selected run, build, explanation setting, and logs.")
for path in score_paths:
    print("Scores:", path.relative_to(automl_dir))
    scores = read_csv(path)
    if "score" in scores.columns:
        scores = scores.sort_values("score", ascending=False)
    display(scores)

In [ ]:
analysis_paths = sorted(automl_dir.rglob(
    "column_analysis/test/column_analysis.json"
)) if automl_dir else []
if not analysis_paths:
    print("No column analysis found for the selected run.")
for path in analysis_paths:
    print("Column analysis:", path.relative_to(automl_dir))
    analysis = read_json(path)
    display(pd.json_normalize(analysis.get("cohorts") or []))

### Locate other diagnostics

When present, `best_trainer.json` identifies the selected trial; `train_progress.json` records individual trial progress; TensorBoard event files support deeper curve inspection; `failed_trials` contains failure diagnostics; and `analyze` contains optimization analysis. Parameter importance in optimization analysis concerns training settings, not input-column explanations. Not every run emits every artifact type.

In [ ]:
if automl_dir:
    for pattern in ["best_trainer.json", "version_*/train_progress.json",
                    "**/events.out.tfevents.*", "failed_trials/*", "analyze/*"]:
        paths = sorted(automl_dir.glob(pattern))
        print(pattern, ":", len(paths), "matches")
        for path in paths[:20]:
            print(" ", path.relative_to(automl_dir))

## Optional: preserve diagnostics

The selected AutoML directory is archived as a ZIP below `ARTIFACT_ROOT/training-results` in the Unity Catalog Volume. Archives can contain large checkpoints and sensitive diagnostic information. This is separate from the retained model bundle and source data.


In [ ]:
uploaded_archive_path = None
if UPLOAD_DIAGNOSTICS and INSPECT_DIAGNOSTICS:
    if automl_dir is None:
        raise ValueError("Select a run in optional diagnostics first.")
    archive_root = ARTIFACT_ROOT / "training-results"
    archive_root.mkdir(parents=True, exist_ok=True)
    archive_name = f"tpch-{automl_dir.name}-{uuid4().hex}"
    # Build the ZIP on local disk first (zipfile needs seek, which FUSE doesn't support)
    local_zip = Path(shutil.make_archive(
        str(Path(tempfile.mkdtemp(prefix="kumo-archive-", dir=scratch_root)) / archive_name), "zip", root_dir=automl_dir,
    ))
    dest = archive_root / local_zip.name
    shutil.copy2(local_zip, dest)
    local_zip.unlink()
    uploaded_archive_path = dest
    print("Archive:", uploaded_archive_path)
else:
    print("Archive upload skipped.")


### Retrieve and inspect an archive

The notebook inspects the ZIP created in this run. To inspect an earlier archive, set `SAVED_ARCHIVE` to its full `/Volumes/.../*.zip` path. The cell lists ZIP contents without extracting or executing them.


In [ ]:
if DOWNLOAD_DIAGNOSTICS:
    from zipfile import ZipFile
    archive_to_inspect = Path(SAVED_ARCHIVE) if SAVED_ARCHIVE else globals().get("uploaded_archive_path")
    if not archive_to_inspect:
        raise ValueError("Set SAVED_ARCHIVE or enable archive creation and select a run first.")
    if not archive_to_inspect.is_file() or archive_to_inspect.suffix != ".zip":
        raise FileNotFoundError("Archive is not a readable ZIP under the Unity Catalog Volume.")
    with ZipFile(archive_to_inspect) as archive:
        members = archive.infolist()
        print("Members:", len(members))
        for member in members[:50]:
            print(member.filename, member.file_size, "bytes")
    print("Archive:", archive_to_inspect)
else:
    print("Archive retrieval skipped.")


## Next steps

Record the retained model-bundle path under the Unity Catalog Volume. This notebook trains and predicts in the same session; it does not load an earlier bundle. To train again, rerun the connector/view setup, graph snapshot, training-table generation, and fit steps. Changing the graph, target, or horizon requires revalidation and may require retraining.

The TPC-H dataset at Scale Factor 10 (750K customers, 7.5M orders, 30M line items) is larger than a typical getting-started example. For faster iteration, consider filtering to a subset of customers or a narrower date range during development.
